## ExPECA Testbed HI Setup

Start with reading README to understand the purpose of this notebook and the setup options.

---

### Overview

**The notebook will create a setup with the following components**

- Edge Device: Start a containerized application that runs the edge device in the ExPECA testbed.
  - Option 1 = Raspberry Pi (arm64)
  - Option 2 = Normal testbed node (amd64)
- Edge Server: Start a containerized application that runs the edge server in the ExPECA testbed.

**Steps to follow:**

1. Setup authentication for the ExPECA testbed.
2. Install required packages and libraries
3. Setup Edge Server
  - Reserve testbed equiment
  - Load and start container
4. Setup Edge Device
  - Select wanted option
  - Reserve testbed equiment
  - Load and start container


### 1. Setup Authentication for the ExPECA Testbed

---

**Steps:**

1. Login to Chameleon and download openrc.sh file from [here](https://testbed.expeca.proj.kth.se/project/api_access/openrc/).
2. Upload it here next to this notebook and continue.
3. Edit path to match your "xxx-project-openrc.sh" file if needed.
4. Enter your ExPECA password when asked.

In [ ]:
import os, re
from getpass import getpass

# Replace with your OpenStack credentials file path
with open('project_name-openrc.sh', 'r') as f:
    script_content = f.read()
    pattern = r'export\s+(\w+)\s*=\s*("[^"]+"|[^"\n]+)'
    matches = re.findall(pattern, script_content)

    for name, value in matches:
        os.environ[name] = value.strip('"')

password = getpass('enter your expeca password:')
os.environ['OS_PASSWORD'] = password

### 2. Install Required Packages and Libraries

---

There might be some warnings, ignore them and test if everything still works.

In [ ]:
# Install required packages
!pip uninstall -q -y moviepy
!pip install -q jedi
!pip install -q git+https://github.com/KTH-EXPECA/python-chi

# Import required libraries
import json, time
from loguru import logger
import chi.network, chi.container, chi.network
from chi.expeca import reserve, list_reservations, unreserve_byid, get_container_status, wait_until_container_removed, show_reservation_byname, restart_sdr, make_sdr_ni, make_sdr_mango, sdr_tools, get_available_publicips, get_segment_ids, get_radio_interfaces, get_worker_interfaces

### 3. Setup Edge Server

---

1. Reserve testbed equiment for the Edge Server
2. Load and run the container for Edge Server (amd64)
3. Print out used public IP for the container

In [ ]:
# Worker reservation for Edge Server
worker_name = "worker-09"

found = False
leaseslist = list_reservations(brief=True)
for lease_dict in leaseslist:
  if lease_dict["name"] == worker_name+"-lease":
    worker_reservation_id = lease_dict["reservation_id"]
    found = True

if not found:
  worker_lease = reserve(
    { "type":"device", "name":worker_name, "duration": { "days":7, "hours":0 } }
  )
  worker_reservation_id = worker_lease["reservations"][0]["id"]

print(json.dumps(leaseslist,indent=4))

In [ ]:
# Edge Server public IP reservation, loading and running container

# 1. Check public IPs and select one
available_pub_ips = get_available_publicips()
pub_ip = available_pub_ips[0]
logger.info(f"Available public ips: {available_pub_ips}.")
logger.info(f"We choose {pub_ip} for this container.")
pub_ip = available_pub_ips[0]

# 2. Check available interfaces of the worker
interfaces = list(get_worker_interfaces(worker_name).values())[0]
available_ifs = []
for interface in interfaces.keys():
  if len(interfaces[interface]['connections']) == 0:
    available_ifs.append(interface)
logger.info(f"Available interfaces on {worker_name}: {available_ifs}")

# 3. Run the container
publicnet = chi.network.get_network("serverpublic")
container_name = "hi-edge-server"
chi.container.create_container(
    name = container_name,
    image = "h3nkk44/hi-framework-edge-server:latest_amd64",
    reservation_id = worker_reservation_id,
    environment = {
        "DNS_IP":"8.8.8.8",
        "GATEWAY_IP":"130.237.11.97",
        "PASS":"expeca"
    },
    mounts = [],
    nets = [
        { "network" : publicnet['id'] },
    ],
    labels = {
        "networks.1.interface":available_ifs[0],
        "networks.1.ip":pub_ip+"/27",
        "networks.1.gateway":"130.237.11.97",
        "capabilities.privileged":"true",
    },
)
chi.container.wait_for_active(container_name)
logger.success(f"created {container_name} container, reachable at {pub_ip}.")

edge_server_pub_ip = pub_ip
logger.success(f"Edge Server public IP: {edge_server_pub_ip}")

### 4. Setup Edge Device

---

*Note: Edge servers public IP is passed as "edge_server_pub_ip" to Edge Device*

**Steps:**

1. Reserve testbed equiment for the Edge Device:
  - Option 1 = Raspberry Pi (amd64)
  - Option 2 = Worker node (arm64)
2. Load and run the container for Edge Device
  - Use same option as previously
3. Print out used public IP for the container

#### Option 1 = Raspberry Pi (arm64)

You can edit the following parameters
- Used worker = "worker_name"
- Used image = "image"

In [ ]:
# Option 1 = Raspberry Pi (arm64)
# Edge Device worker reservation

worker_name = "worker-21"

found = False
leaseslist = list_reservations(brief=True)
for lease_dict in leaseslist:
  if lease_dict["name"] == worker_name+"-lease":
    worker_reservation_id = lease_dict["reservation_id"]
    found = True

if not found:
  worker_lease = reserve(
    { "type":"device", "name":worker_name, "duration": { "days":7, "hours":0 } }
  )
  worker_reservation_id = worker_lease["reservations"][0]["id"]

print(json.dumps(leaseslist,indent=4))

In [ ]:
# Option 1 = Raspberry Pi (arm64)
# Edge Device public IP reservation, loading and running container

# 1. Check public IPs and select one
available_pub_ips = get_available_publicips()
pub_ip = available_pub_ips[0]
logger.info(f"Available public ips: {available_pub_ips}.")
logger.info(f"We choose {pub_ip} for this container.")
pub_ip = available_pub_ips[0]

# 2. Check available interfaces of the worker
interfaces = list(get_worker_interfaces(worker_name).values())[0]
available_ifs = []
for interface in interfaces.keys():
  if len(interfaces[interface]['connections']) == 0:
    available_ifs.append(interface)
logger.info(f"Available interfaces on {worker_name}: {available_ifs}")

# 3. Run the container
publicnet = chi.network.get_network("serverpublic")
container_name = "hi-edge-device"
chi.container.create_container(
    name = container_name,
    image = "h3nkk44/hi-framework-edge-device:latest_arm64",
    reservation_id = worker_reservation_id,
    environment = {
        "DNS_IP":"8.8.8.8",
        "GATEWAY_IP":"130.237.11.97",
        "PASS":"expeca",
        "EDGE_SERVER_IP": edge_server_pub_ip
    },
    mounts = [],
    nets = [
        { "network" : publicnet['id'] },
    ],
    labels = {
        "networks.1.interface":available_ifs[0],
        "networks.1.ip":pub_ip+"/27",
        "networks.1.gateway":"130.237.11.97",
        "capabilities.privileged":"true",
    },
)
chi.container.wait_for_active(container_name)
logger.success(f"created {container_name} container, reachable at {pub_ip}.")
logger.success(f"Edge Device public IP: {pub_ip}")

#### Option 2 = Worker node (amd64)

You can edit the following parameters
- Used worker = "worker_name"
- Used image = "image"

In [ ]:
# Option 2 = Worker node (amd64)
# Edge Device worker reservation

worker_name = "worker-09"

found = False
leaseslist = list_reservations(brief=True)
for lease_dict in leaseslist:
  if lease_dict["name"] == worker_name+"-lease":
    worker_reservation_id = lease_dict["reservation_id"]
    found = True

if not found:
  worker_lease = reserve(
    { "type":"device", "name":worker_name, "duration": { "days":7, "hours":0 } }
  )
  worker_reservation_id = worker_lease["reservations"][0]["id"]

print(json.dumps(leaseslist,indent=4))

In [ ]:
# Option 2 = Server node (amd64)
# Edge Device public IP reservation, loading and running container

# 1. Check public IPs and select one
available_pub_ips = get_available_publicips()
pub_ip = available_pub_ips[0]
logger.info(f"Available public ips: {available_pub_ips}.")
logger.info(f"We choose {pub_ip} for this container.")
pub_ip = available_pub_ips[0]

# 2. Check available interfaces of the worker
interfaces = list(get_worker_interfaces(worker_name).values())[0]
available_ifs = []
for interface in interfaces.keys():
  if len(interfaces[interface]['connections']) == 0:
    available_ifs.append(interface)
logger.info(f"Available interfaces on {worker_name}: {available_ifs}")

# 3. Run the container
publicnet = chi.network.get_network("serverpublic")
container_name = "hi-edge-device"
chi.container.create_container(
    name = container_name,
    image = "h3nkk44/hi-framework-edge-device:latest_amd64",
    reservation_id = worker_reservation_id,
    environment = {
        "DNS_IP":"8.8.8.8",
        "GATEWAY_IP":"130.237.11.97",
        "PASS":"expeca",
        "EDGE_SERVER_IP": edge_server_pub_ip
    },
    mounts = [],
    nets = [
        { "network" : publicnet['id'] },
    ],
    labels = {
        "networks.1.interface":available_ifs[0],
        "networks.1.ip":pub_ip+"/27",
        "networks.1.gateway":"130.237.11.97",
        "capabilities.privileged":"true",
    },
)
chi.container.wait_for_active(container_name)
logger.success(f"created {container_name} container, reachable at {pub_ip}.")
logger.success(f"Edge Server public IP: {edge_server_pub_ip}")

### Connecting to the Edge Device and Edge Server using SSH

---

Now the setup should be completed and you can use the external controller scripts to run the experiments.
- External controller scripts are found in `app/external_controllers/`
- SSH is not required if using public IP's.
- Chameleon web interface allows easy monitoring and control of the containers.

You can ssh into the containers with the following command:
```
ssh root@[public_ip_address]
```

### Additional Guides

---

**1. Starting the Experiment:**

1. Open ExPECA Web GUI.
2. Open Edge Device container console.
 - Check if folder or old results exists (should be automatically deleted):
```bash
ls /app/results/
```
 - If not run:
```bash
rm -rf /app/results/*
```

3. Open Edge Server container console.
 - Check if folder exist or old results (should be automatically deleted):
```bash
ls /app/results/
```
 - If not run:
```bash
rm -rf /app/results/*
```

4. Start experiment script in the background with tmux:
```bash
apt update
apt install tmux
tmux new -s mysession /app/start.sh
```
You can reconnect to the tmux session with:

```bash
tmux attach -t mysession
```

4. Start experiment script with:
```bash	
/app/start.sh
```

5. Enter wanted datasets path (data/datasets/...):
```bash
imagenetV2/matched-frequency-format-val/
imagenetV2/test_subset/
imagenette/val_renamed/
```

6. Select other parameters to be used for the experiment.
7. Don't close the console or web GUI or it will stop the experiment.
 - The console will show experiment finished message.

**2. Collecting Results and Logs:**

1. Open ExPECA Web GUI.
2. Open Edge Device console.
3. Insert command (set correct Edge Server public IP):
    - Sends the results to the Edge Server from the Edge Device.

```bash
scp -v -r /app/results/ root@130.237.11.119:/app/
```

4. Open terminal on your local machine.
5. Insert command (set correct Edge Server public IP):
```bash
scp -v -r root@130.237.11.119:/app/results/ C:/Users/henri/Downloads/run_001
```